# Módulo 1: Análisis Exploratorio de Datos (EDA) y Preparación de Incidentes

**Proyecto Integrador - Maestría en Inteligencia Artificial Aplicada**  
**Sistema Asignador de Incidentes TI (ServiceNow)**

Este cuaderno realiza el análisis exploratorio exhaustivo, auditoría de calidad, consolidación de variables y resolución de ambigüedades sobre los tickets históricos de incidentes bancarios.  
Al finalizar, exporta el conjunto de datos limpio y curado `Incidentes_Preparados.csv`, que sirve como punto de partida estandarizado para los cuadernos de modelado (**02 Línea Base TF-IDF** y **03 Arquitectura HNLP-MC**).

## 1. Carga de Librerías y Configuración Inicial

In [ ]:
# ── Librería estándar ─────────────────────────────────────────────────────────
import os
import re
import unicodedata
import warnings
from collections import Counter

# ── Datos y visualización ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings('ignore')

# ── Procesamiento de texto ────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords

# ── Configuración de gráficos ─────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set(rc={'figure.figsize': (10, 6)})


## 2. Carga y Exploración Inicial de Datos

In [ ]:
# Cargar el archivo CSV
# Ruta relativa desde Analisis/ hacia Data/ (un nivel arriba)
ruta_csv = "../Datos/IncidentesCategorizados_v2.csv"
print('Directorio actual:', os.getcwd())
print('Ruta CSV:', ruta_csv)

# Intenta cargar el archivo
try:
    try:
        df = pd.read_csv(ruta_csv, sep=",", encoding="latin-1")
        if len(df.columns) == 1:
            raise ValueError("Wrong delimiter")
    except (ValueError, pd.errors.ParserError):
        df = pd.read_csv(ruta_csv, sep=";", encoding="latin-1")
except FileNotFoundError as e:
    print('No se encontró el archivo. Verifica la ruta y ubicación.')
    raise e

# Mostrar primeras filas y resumen
print('Dimensiones:', df.shape)
df.head()


## 3. Consolidación de Aplicativo de Negocio (`cmdb_ci_business_app` y `cmdb_ci`)

Ambas columnas corresponden al mismo concepto en ServiceNow que cambió de nombre históricamente.  
Se combinan mediante `combine_first()` para recuperar una cobertura del 94% sobre los incidentes.

In [ ]:
# Cominando cmdb_ci_business_app y cmdb_ci dado que corresponden al mismo campo
# 1. Asegurar que cadenas vacías o espacios se interpreten como NaN
df['cmdb_ci_business_app'] = df['cmdb_ci_business_app'].replace(r'^\s*$', np.nan, regex=True)
df['cmdb_ci'] = df['cmdb_ci'].replace(r'^\s*$', np.nan, regex=True)

# 2. Rellenar los valores nulos de cmdb_ci_business_app con los de cmdb_ci
df['cmdb_ci_business_app'] = df['cmdb_ci_business_app'].combine_first(df['cmdb_ci'])

# 3. Eliminar la columna antigua redundante
df = df.drop(columns=['cmdb_ci'])

In [ ]:
# Tipos de datos y valores nulos
df.info()
df.isnull().sum().sort_values(ascending=False).head(15)

In [ ]:
# Exploración rápida de variables informativas
print('\nPorcentaje de valores nulos por columna:')
print((df.isnull().mean()*100).sort_values(ascending=False))
print('\nNúmero de valores únicos por columna:')
print(df.nunique().sort_values(ascending=False))

# Sugerencia de variables informativas (no ID, no columnas vacías o con un solo valor)
columnas_informativas = [col for col in df.columns if df[col].nunique()>1 and df[col].nunique()<df.shape[0]*0.9 and not col.lower().startswith('id')]
print('\nColumnas potencialmente informativas:')
print(columnas_informativas)


## 4. Limpieza y Normalización de la Variable Objetivo (`Clasificación`)

Se homogenizan etiquetas (mayúsculas/minúsculas, singular/plural, sin tildes) y se identifican los tickets sin clasificar.

In [ ]:
# Cambiar el nombre de la columna 'Categoria' por 'Clasificación'
df = df.rename(columns={"Categoría": "Clasificación"})

In [ ]:
# Mostrar la distribución de clases (conteo de tickets por clasificación)
conteo_clases = df['Clasificación'].value_counts(dropna=False)
print(conteo_clases)

In [ ]:
# Revisar valores únicos y nulos en 'Clasificación'
if 'Clasificación' in df.columns:
    print('Valores únicos en Clasificación:', df['Clasificación'].unique())
    print('Valores nulos en Clasificación:', df['Clasificación'].isnull().sum())
else:
    print("La columna 'Clasificación' no está presente en el DataFrame.")

In [ ]:
# Normalización de valores en la columna 'Clasificación'
def normalizar_texto(texto):
    if pd.isnull(texto):
        return "sin_clasificar"
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.replace('.', '')
    texto = texto.replace('/', '-')
    texto = texto.strip()
    return texto

df['Clasificación'] = df['Clasificación'].apply(normalizar_texto)

# Verifica los valores únicos después de la normalización
# Revisar valores únicos y nulos en 'Clasificación'
if 'Clasificación' in df.columns:
    print('Valores únicos en Clasificación:', df['Clasificación'].unique())
    print('Valores nulos en Clasificación:', df['Clasificación'].isnull().sum())
else:
    print("La columna 'Clasificación' no está presente en el DataFrame.")


In [ ]:
# 1. Agrupar valores en plural y singular en la columna 'Clasificación'
agrupaciones_plural = {
    'cheques': 'cheque',
    'cuentas': 'cuenta',
    'activos': 'activo',
    'garantias': 'garantia',
    'inversiones': 'inversion',
    'sobregiros': 'sobregiro',
    'usuarios': 'usuario',
    'procesos': 'proceso',
    'clientes': 'cliente',
    'reportes online': 'reporte online',
    'reportes batch': 'reporte batch',
    'formas numeradas': 'forma numerada',
    # Agregar más se detectan otros casos
}
df['Clasificación'] = df['Clasificación'].replace(agrupaciones_plural)

In [ ]:
# Reemplazar valores sin clasificar y pendientes → sin_clasificar
df['Clasificación'] = df['Clasificación'].replace(
#    ['0', 0, None, np.nan, 'no aplica', 'pendiente'],
    ['0', 0, None, np.nan, 'pendiente'],
    'sin_clasificar'
)

# Eliminar filas con clases que tienen muy pocos ejemplos para entrenar (< 3) o son irrelevantes
clases_a_eliminar = ['cc-cancelacion batch', 'cc-evento-alerta', 'turno']
filas_antes = len(df)
df = df[~df['Clasificación'].isin(clases_a_eliminar)].reset_index(drop=True)
print(f"Filas eliminadas: {filas_antes - len(df)}  ({clases_a_eliminar})")
print(f"Total filas restantes: {len(df)}")


In [ ]:
# Verifica los valores únicos después de la normalización
print(df['Clasificación'].unique())


In [ ]:
# Mostrar la distribución de clases (conteo de tickets por clasificación)
conteo_clases = df['Clasificación'].value_counts(dropna=False)
print(f"Total de clases de Clasificación: {conteo_clases.shape[0]}")
print()
print(conteo_clases)


In [ ]:
# Verificar duplicados en el DataFrame
duplicados = df[df.duplicated()]
print(f"Número de filas duplicadas: {duplicados.shape[0]}")
if not duplicados.empty:
    display(duplicados.head())
else:
    print("No se encontraron filas duplicadas.")

## 5. Análisis de Relevancia y Diagnóstico de Variables para el Asignador

Se evalúa la capacidad discriminante real de cada columna usando el coeficiente **$V$ de Cramér** (corregido por sesgo).  
Se clasifican las variables entre: útiles para el clasificador, descartadas por varianza casi nula, o excluidas por **Fuga de Información (*Data Leakage*)**.

In [ ]:
# ==============================================================================
# ANÁLISIS INTEGRAL DE RELEVANCIA DE VARIABLES PARA LA CLASIFICACIÓN DE TICKETS
# ==============================================================================
# Evalúa todas las columnas del conjunto de datos para identificar cuáles aportan
# poder predictivo real y cuáles deben descartarse (por varianza nula o data leakage).

from scipy.stats import chi2_contingency

# 1. Filtrar registros etiquetados válidos para medir asociación real
mask_valido = ~df['Clasificación'].astype(str).str.strip().str.lower().isin(
    ['0', 'nan', 'sin_clasificar', 'pendiente', 'none', '']
) & df['Clasificación'].notnull()

df_eval = df[mask_valido].copy()
print(f"Evaluando relevancia predictiva sobre {len(df_eval)} tickets clasificados ({df_eval['Clasificación'].nunique()} categorías).\n")

# Función para calcular V de Cramér con corrección de sesgo (Bergsma, 2013)
def cramers_v_corrected(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.shape[0] <= 1 or confusion_matrix.shape[1] <= 1:
        return 0.0
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2corr / denom))

# 2. Clasificación funcional de columnas según el ciclo de vida del ticket
columnas_fuga = ['close_notes', 'resolved_at', 'business_duration', 'reopen_count', 'state', 'assigned_to']
columnas_id = ['number', 'sys_created_on', 'caller_id', 'u_affected_user']
columnas_texto = ['short_description', 'description', 'comments_and_work_notes']

columnas_categoricas = [
    col for col in df.columns 
    if col not in columnas_fuga + columnas_id + columnas_texto + ['Clasificación']
    and df[col].nunique() > 1
]

# 3. Evaluación de variables categóricas candidatas
resultados = []
for col in columnas_categoricas:
    s_full = df[col]
    s_eval = df_eval[col].fillna('DESCONOCIDO').astype(str)
    
    nunique = s_full.nunique(dropna=False)
    pct_null = s_full.isnull().mean() * 100
    top_val_pct = (s_full.value_counts(normalize=True, dropna=False).iloc[0]) * 100 if len(s_full) > 0 else 0
    
    cv = cramers_v_corrected(s_eval, df_eval['Clasificación'])
    
    # Diagnóstico y utilidad
    if top_val_pct >= 95.0:
        utilidad = "Descartar"
        motivo = "Cuasi-constante (Varianza nula)"
    elif cv >= 0.35:
        utilidad = "Alta"
        motivo = "Fuerte asociación con Clasificación"
    elif cv >= 0.20:
        utilidad = "Media"
        motivo = "Moderada asociación discriminante"
    else:
        utilidad = "Baja"
        motivo = "Poca capacidad discriminante"
        
    resultados.append({
        'Variable': col,
        'Nulos (%)': round(pct_null, 1),
        'Valores Únicos': nunique,
        'Dominancia Moda (%)': round(top_val_pct, 1),
        "Cramér's V": round(cv, 4),
        'Utilidad': utilidad,
        'Diagnóstico': motivo
    })

df_ranking = pd.DataFrame(resultados).sort_values(by="Cramér's V", ascending=False).reset_index(drop=True)

print("=" * 85)
print("1. RANKING DE RELEVANCIA ESTADÍSTICA (VARIABLES CATEGÓRICAS)")
print("=" * 85)
print(df_ranking[['Variable', 'Nulos (%)', 'Dominancia Moda (%)', "Cramér's V", 'Utilidad', 'Diagnóstico']].to_string(index=False))

# 4. Diagnóstico de campos de texto (NLP)
print("\n" + "=" * 85)
print("2. CAMPOS DE TEXTO (PRINCIPAL FUENTE DE INFORMACIÓN PARA EL CLASIFICADOR)")
print("=" * 85)
for col in columnas_texto:
    if col in df.columns:
        pct_nulos = df[col].isnull().mean() * 100
        avg_len = df[col].fillna('').astype(str).str.len().mean()
        estado = "RECOMENDADA (Core del modelo NLP)" if pct_nulos < 10 else "Opcional (Muchos nulos / Notas internas)"
        print(f"• {col:<25}: {pct_nulos:5.1f}% nulos | Longitud media: {avg_len:5.1f} caracteres -> {estado}")

# 5. Advertencia de Fuga de Información
print("\n" + "=" * 85)
print("3. VARIABLES EXCLUIDAS POR FUGA DE DATOS (DATA LEAKAGE)")
print("=" * 85)
print("Las siguientes columnas NO deben usarse si el modelo clasifica tickets al crearse (estado 'Nuevo'):")
for col in columnas_fuga:
    if col in df.columns:
        print(f"⚠ {col:<25}: Se genera o modifica durante/después de la gestión del incidente.")

# 6. Gráficos de Alto Valor
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico A: Barras de Cramér's V ordenadas por utilidad
paleta = {'Alta': '#2ca02c', 'Media': '#ff7f0e', 'Descartar': '#d62728', 'Baja': '#7f7f7f'}
sns.barplot(
    data=df_ranking,
    x="Cramér's V",
    y='Variable',
    hue='Utilidad',
    palette=paleta,
    dodge=False,
    ax=axes[0]
)
axes[0].set_title("Poder Discriminante de Variables Categóricas (V de Cramér)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("V de Cramér (0 = Sin relación, 1 = Relación perfecta)")
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(title="Utilidad")

# Gráfico B: Proporción de Clases en la Variable Categórica Top (cmdb_ci_business_app)
top_col = 'cmdb_ci_business_app' if 'cmdb_ci_business_app' in df_eval.columns else df_ranking.iloc[0]['Variable']
top_categorias = df_eval['Clasificación'].value_counts().head(8).index
df_top_clases = df_eval[df_eval['Clasificación'].isin(top_categorias)]

# Frecuencias relativas (% por fila) para ver la mezcla de categorías en cada aplicación
top_apps = df_top_clases[top_col].value_counts().head(8).index
crosstab_norm = pd.crosstab(
    df_top_clases[df_top_clases[top_col].isin(top_apps)][top_col],
    df_top_clases['Clasificación'],
    normalize='index'
) * 100

sns.heatmap(crosstab_norm, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': '% de la aplicación'}, ax=axes[1])
axes[1].set_title(f"Distribución Relativa (% fila): '{top_col}' vs Top Clases", fontsize=12, fontweight='bold')
axes[1].set_ylabel(top_col)
axes[1].set_xlabel("Clasificación")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 6. Enriquecimiento Textual y Resolución de Ambigüedades Humanas

1. Se unifica el texto incorporando los metadatos de mayor valor (`short_description`, `description`, `cmdb_ci_business_app`, `u_subcategory_2`, `u_subcategory`, `title`, `company`, `department`).
2. Se detectan tickets idénticos que fueron clasificados con diferentes categorías por analistas humanos.
3. Se resuelve la ambigüedad aplicando la regla mayoritaria para eliminar ruido de etiquetas.
4. Se preservan **todos** los registros (sin borrar duplicados) para mantener la frecuencia real del negocio.

In [ ]:
# Unificar texto relevante para análisis de similitud y modelado NLP
# Enriquecemos el texto con los metadatos de alto poder predictivo identificados en el EDA:
# - cmdb_ci_business_app (aplicación bancaria consolidada con cmdb_ci)
# - u_subcategory y u_subcategory_2 (subcategorías técnicas del incidente)
# - u_affected_user.title (cargo o rol del usuario solicitante)
# - u_affected_user.company (empresa: Banco Pichincha vs Tata Consultancy Services)
# - u_affected_user.department (departamento o área del usuario)

df['texto_unificado'] = (
    df['short_description'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['cmdb_ci_business_app'].fillna('') + ' ' +
    df['u_subcategory'].fillna('') + ' ' +
    df['u_subcategory_2'].fillna('') + ' ' +
    df['u_affected_user.title'].fillna('') + ' ' +
    df['u_affected_user.company'].fillna('') + ' ' +
    df['u_affected_user.department'].fillna('')
)

# Guardar copia del texto crudo ANTES de limpiar (para análisis de duplicados y GroupSplit)
df['texto_unificado_raw'] = df['texto_unificado']

print(f"✓ 'texto_unificado' generado con éxito para {len(df)} registros.")
print(f"  Longitud promedio del texto enriquecido: {df['texto_unificado'].str.len().mean():.1f} caracteres.")


In [ ]:
# ── Análisis de duplicados exactos en texto_unificado (PRE-limpieza) ──────────
# Usa texto_unificado_raw para que sea robusto sin importar el orden de ejecución.

dup_mask_raw = df['texto_unificado_raw'].duplicated(keep=False)
df_dups_raw = df[dup_mask_raw].copy()

print(f"Tickets con texto_unificado duplicado (crudo): {dup_mask_raw.sum()}  "
      f"({dup_mask_raw.mean():.1%} del total)")
print(f"Textos únicos que se repiten: {df_dups_raw['texto_unificado_raw'].nunique()}")

# Tabla de frecuencias: cuántas veces se repite cada texto y sus clasificaciones
resumen_dups = (
    df_dups_raw
    .groupby('texto_unificado_raw', as_index=False)
    .agg(
        n_tickets       = ('number', 'count'),
        clasificaciones = ('Clasificación', lambda x: ', '.join(sorted(x.unique()))),
        n_clases        = ('Clasificación', 'nunique'),
        ejemplos        = ('number', lambda x: ', '.join(x.head(4).astype(str)))
    )
    .rename(columns={'texto_unificado_raw': 'texto_unificado'})
    .sort_values('n_tickets', ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 textos más repetidos:")
with pd.option_context('display.max_colwidth', 80):
    display(resumen_dups.head(20))

# Duplicados con clasificaciones distintas (ambigüedad real)
ambiguos = resumen_dups[resumen_dups['n_clases'] > 1]
print(f"\nTextos duplicados con DISTINTAS clasificaciones (ambigüedad): {len(ambiguos)}")
if len(ambiguos):
    with pd.option_context('display.max_colwidth', 80):
        display(ambiguos.head(20))


In [ ]:
# ── IDs de tickets ambiguos (ninguna clasificación es "sin_clasificar") ───────
textos_ambiguos_reales = resumen_dups[
    (resumen_dups['n_clases'] > 1) &
    (~resumen_dups['clasificaciones'].str.contains(r'\bsin_clasificar\b', na=False))
]['texto_unificado'].tolist()

ids_ambiguos = (
    df[df['texto_unificado_raw'].isin(textos_ambiguos_reales)]['number']
    .sort_values()
    .reset_index(drop=True)
)

print(f"Total IDs para revisión: {len(ids_ambiguos)}")
display(ids_ambiguos.to_frame())


In [ ]:
# ── Contenido completo de los tickets ambiguos para revisión ──────────────────
cols_mostrar = ['number', 'texto_unificado_raw', 'Clasificación']
# Agrega columnas extra si existen en df
for col_extra in ['short_description', 'description', 'Subcategoría', 'category', 'subcategory']:
    if col_extra in df.columns and col_extra not in cols_mostrar:
        cols_mostrar.append(col_extra)

df_revision = (
    df[df['number'].isin(ids_ambiguos)]
    [cols_mostrar]
    .sort_values('number')
    .reset_index(drop=True)
)

print(f"Tickets para revisión: {len(df_revision)}")
with pd.option_context('display.max_colwidth', 200, 'display.max_rows', None):
    display(df_revision)


In [ ]:
mapa_mayoritario = {}
for texto in textos_ambiguos_reales:
    conteo = (
        df[df['texto_unificado_raw'] == texto]['Clasificación']
        .value_counts()
    )
    mapa_mayoritario[texto] = conteo.idxmax()

mask_ambiguos = df['texto_unificado_raw'].isin(textos_ambiguos_reales)
df.loc[mask_ambiguos, 'Clasificación'] = df.loc[mask_ambiguos, 'texto_unificado_raw'].map(mapa_mayoritario)

print(f"Tickets corregidos: {mask_ambiguos.sum()}")
print("\nClasificación asignada por texto ambiguo:")
for texto, clase in mapa_mayoritario.items():
    print(f" → '{texto[:80]}...' ► {clase}")

verificacion = (
    df[mask_ambiguos]
    .groupby('texto_unificado_raw')['Clasificación']
    .nunique()
)
assert (verificacion == 1).all(), "¡Aún quedan textos con múltiples clases!"
print(f"\n✓ Verificación OK: todos los textos ambiguos tienen ahora una única clasificación.")


In [ ]:
# ── Preservación de registros y preparación para Group Split ────────────────
# Tras resolver ambigüedades con la clase mayoritaria, se conservan TODOS los registros.
# Las plantillas y textos recurrentes representan la distribución real de incidentes del banco.
# Para evitar Data Leakage (que un mismo texto esté en Train y en Test),
# la separación se realizará mediante GroupShuffleSplit agrupando por 'texto_unificado_raw'.

total_tickets = len(df)
textos_unicos = df['texto_unificado_raw'].nunique()
tickets_repetidos = total_tickets - textos_unicos

print(f"Total de tickets conservados : {total_tickets} (100% de la información preservada)")
print(f"Textos únicos de incidentes  : {textos_unicos}")
print(f"Tickets con textos recurrentes: {tickets_repetidos} ({tickets_repetidos / total_tickets:.1%})")
print(f"-> Nota: No se descarta ningún ticket. El aislamiento train/test se garantizará mediante GroupShuffleSplit.")

print(f"\nDistribución de clases (conteo total de incidentes):")
print(df['Clasificación'].value_counts().to_string())


In [ ]:
# Cargar stopwords ANTES de limpiar_texto para poder usarlas en la limpieza
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))
print(f"✓ Stopwords cargadas: {len(spanish_stopwords)} palabras")

In [ ]:
def limpiar_texto(texto):
    if pd.isnull(texto):
        return ""
    texto = str(texto).lower()
    # Normalizar acentos y caracteres especiales
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    # Eliminar caracteres no alfanuméricos (excepto espacios)
    texto = re.sub(r'[^a-zA-Z0-9\s]', ' ', texto)
    # Eliminar tokens puramente numéricos (códigos, fechas, horas, IDs)
    texto = re.sub(r'\b\d+\b', ' ', texto)
    # Eliminar tokens alfanuméricos que sean mayormente números (ej: ws0067, v1, v2)
    texto = re.sub(r'\b[a-z]*\d+[a-z0-9]*\b', ' ', texto)
    # Eliminar tokens muy cortos (1-2 caracteres) que no aportan significado
    texto = re.sub(r'\b\w{1,2}\b', ' ', texto)
    # Normalizar espacios
    texto = re.sub(r'\s+', ' ', texto).strip()
    # Eliminar stopwords en español directamente en el texto
    tokens = texto.split()
    tokens = [t for t in tokens if t not in spanish_stopwords]
    return ' '.join(tokens)

df['texto_unificado'] = df['texto_unificado'].apply(limpiar_texto)

# Verificar mejora — mostrar muestra de textos limpios
print("Ejemplos de texto limpio (sin stopwords):")
for t in df['texto_unificado'].dropna().sample(5, random_state=42):
    print(" →", t[:120])


## 7. Exportación del Conjunto de Datos Preparado

Se exporta el dataset final curado a `../Datos/Incidentes_Preparados.csv`.  
Este archivo contiene las etiquetas consolidadas, aplicativos unificados y el `texto_unificado_raw`, listo para ser consumido por los cuadernos de entrenamiento **02 (TF-IDF Baseline)** y **03 (Arquitectura HNLP-MC)**.

In [ ]:
# Guardar el DataFrame curado en la carpeta Datos/
ruta_salida_datos = os.path.normpath(os.path.join(os.getcwd(), '..', 'Datos', 'Incidentes_Preparados.csv'))
os.makedirs(os.path.dirname(ruta_salida_datos), exist_ok=True)

df.to_csv(ruta_salida_datos, sep=';', encoding='latin-1', index=False)

print(f"✓ Dataset preparado exportado exitosamente:")
print(f"  Ruta: {ruta_salida_datos}")
print(f"  Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"  Tickets clasificados: {(df['Clasificación'] != 'sin_clasificar').sum()}")
print(f"  Tickets sin clasificar: {(df['Clasificación'] == 'sin_clasificar').sum()}")
